Cloud-only data dependency: this notebook expects access to the PROFILE_DATA clinical-note parquets, and either Azure AD credentials (`dfci_gpt`) or Application Default Credentials with Vertex AI permission (`vertex_ai`), depending on `PROVIDER` below.

# Metastatic Prostate Cancer Label + First-Mention Date

Runs the metastasis diagnosis pipeline through either provider, selected with the `PROVIDER` toggle
in the parameters cell.

It answers one question per patient — *does the record establish metastatic prostate cancer, and
when was it first documented?* — and errs toward "no" whenever the evidence is negated, hedged,
benign/degenerative, regional pelvic nodal only, or attributable to a non-prostate primary.

Two boundaries carry most of the precision risk, and both are enforced deterministically in
`tasks/met_diagnosis/veto.py` rather than left to the model:

- **M1 vs. N1.** Regional pelvic nodal disease alone is *not* metastatic, even though prostatectomy
  pathology describes it with the word "metastatic" ("metastatic adenocarcinoma in 2 of 14 pelvic
  lymph nodes"). Treating it as M1 would mislabel a large number of locally advanced patients.
- **Destination vs. primary.** "Metastatic prostate cancer to the lung" is the target finding;
  "known metastatic lung primary" is not.

Definitive imaging impressions qualify and rank equally with pathology and clinician assertions;
hedged radiology ("suspicious for", "concerning for", "cannot exclude") does not, no matter how
strongly worded the rest of the report is.

Dates are earliest-documentation: a date stated in the text when one exists, otherwise the earliest
qualifying note date, with `date_source` / `date_precision` recording which. The note-date path is
the common one here, not an edge case — metastatic disease is rarely given an explicit date.

Pipeline steps:
1. Scan the merged PROFILE_DATA pathology, imaging, and progress-note parquets into an evidence Parquet (provider-independent).
2. Run `tasks/met_diagnosis/build_met_dx_labels.py --provider PROVIDER` to map candidates, adjudicate each patient, and write the labels.

All run toggles default to `False`; review the printed commands and paths before enabling a step.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
from IPython.display import display


def find_repo_root(start):
    """Find the repo root (contains pyproject.toml) from the notebook directory."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the repo root (pyproject.toml)")


REPO_ROOT = find_repo_root(Path.cwd())
PYTHON = sys.executable


def shell_join(parts):
    return " ".join(shlex.quote(str(part)) for part in parts)


def run_command(parts, env=None):
    print(shell_join(parts))
    process = subprocess.Popen(
        parts, cwd=REPO_ROOT, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, bufsize=0,
    )
    progress_display = None
    buffer = []
    carriage_line = False

    def show_progress(line):
        nonlocal progress_display
        payload = {"text/plain": line}
        if progress_display is None:
            progress_display = display(payload, raw=True, display_id=True)
        else:
            progress_display.update(payload, raw=True)

    while True:
        char = process.stdout.read(1)
        if char == b"":
            break
        if char == b"\r":
            if buffer:
                show_progress(b"".join(buffer).decode("utf-8", errors="replace"))
            buffer = []
            carriage_line = True
        elif char == b"\n":
            line = b"".join(buffer).decode("utf-8", errors="replace")
            if carriage_line:
                show_progress(line)
                progress_display = None
            else:
                print(line)
            buffer = []
            carriage_line = False
        else:
            buffer.append(char)
    if buffer:
        (show_progress if carriage_line else print)(b"".join(buffer).decode("utf-8", errors="replace"))
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f"Command failed with exit code {returncode}")


def build_run_env(provider):
    """Child-process environment for the selected provider. Existing environment
    values are preserved except for the explicit notebook parameters below."""
    env = os.environ.copy()
    if provider == "vertex_ai":
        env["VERTEX_PROJECT"] = VERTEX_PROJECT
        env["VERTEX_LOCATION"] = VERTEX_LOCATION
    return env


DEFAULT_DATA_ROOT = Path(
    os.environ.get("LLM_ANNOTATIONS_DATA_PATH", "/data/gusev/USERS/jpconnor/data/LLM_annotations/")
)
PROFILE_DATA_ROOT = Path(os.environ.get("PROFILE_DATA_PATH", "/data/gusev/USERS/jpconnor/data/PROFILE_DATA/"))
DEFAULT_NOTES_PARQUETS = [PROFILE_DATA_ROOT / "CLINICAL_NOTES" / name for name in ("PATHOLOGY_NOTES.parquet", "IMAGING_NOTES.parquet", "PROGRESS_NOTES.parquet")]

print(f"Repo root: {REPO_ROOT}")


In [ ]:
# Provider toggle — everything below adapts to this.
PROVIDER = "vertex_ai"          # or "dfci_gpt"

MODEL_NAME = {
    "dfci_gpt":  "gpt-4o",
    "vertex_ai": "gemini-2.5-flash-lite",
}[PROVIDER]

# Vertex AI settings (unused when PROVIDER == "dfci_gpt")
VERTEX_PROJECT = "gusevlabllm"
VERTEX_LOCATION = "us-central1"

MRNS = []
MRN_FILE = "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/mrn_lists/adt_mrns.csv"
NOTES_PARQUET_PATHS = DEFAULT_NOTES_PARQUETS

OUTPUT_DIR = DEFAULT_DATA_ROOT / "LLM_met_diagnosis"
EVIDENCE_PATH = OUTPUT_DIR / "met_dx_evidence.parquet"

# Collection settings
NOTE_TYPES = None             # e.g. ["Imaging"] — restrict scanning to these NOTE_TYPE values
# Wider than the NEPC dx collector's 1200: radiology negation spans are long
# ("no new osseous lesions ... unchanged from prior ... no evidence of metastatic
# disease"), and site attribution often sits a sentence away from the term.
CONTEXT_CHARS = 1500
PAYLOAD_MAX_CHARS = 60000
OVERWRITE_EVIDENCE = False

# LLM extraction settings
MAX_WORKERS = 16
MAX_RETRIES = 3
LIMIT_PATIENTS = None
OVERWRITE = False

# Step toggles
RUN_COLLECT_EVIDENCE = False  # required before extraction
RUN_EXTRACTION = False
RUN_RETRY_FAILURES = False    # rerun only incomplete chunk maps/patient syntheses

RUN_ENV = build_run_env(PROVIDER)


In [ ]:
# True rebuilds the saved candidate/audit/label Parquets without LLM calls or failure retries.
REBUILD_LABELS_ONLY = False


In [ ]:
notes_parquet_status = {path: Path(path).exists() for path in NOTES_PARQUET_PATHS}
evidence_exists = Path(EVIDENCE_PATH).exists()
labels_path = Path(OUTPUT_DIR) / "met_dx_labels.parquet"
labels_exists = labels_path.exists()

print(f"Provider: {PROVIDER}")
print(f"Model: {MODEL_NAME}")
print(f"Parallel workers: {MAX_WORKERS}")
print("PROFILE_DATA note parquets:")
for path, exists in notes_parquet_status.items():
    print(f"  {exists}: {path}")
print(f"Evidence Parquet exists: {evidence_exists}")
print(f"Path: {EVIDENCE_PATH}")
print(f"Labels Parquet exists: {labels_exists}")
print(f"Path: {labels_path}")

In [ ]:
missing_note_parquets = [path for path, exists in notes_parquet_status.items() if not exists]
if missing_note_parquets:
    raise FileNotFoundError(f"Missing PROFILE_DATA note parquets: {missing_note_parquets}")
print("PROFILE_DATA note inputs are ready.")

In [ ]:
print("Using PROFILE_DATA parquet notes directly; no note compilation step is needed.")

In [ ]:
collect_evidence_cmd = [
    PYTHON,
    "preprocessing/cli/collect_met_dx_notes.py",
    "--output-dir",
    OUTPUT_DIR,
    "--context-chars",
    str(CONTEXT_CHARS),
    "--payload-max-chars",
    str(PAYLOAD_MAX_CHARS),
]
for notes_parquet_path in NOTES_PARQUET_PATHS:
    collect_evidence_cmd.extend(["--notes-parquet", notes_parquet_path])

if MRNS:
    collect_evidence_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
if MRN_FILE is not None:
    collect_evidence_cmd.extend(["--mrn-file", MRN_FILE])
if NOTE_TYPES:
    collect_evidence_cmd.extend(["--note-types", *NOTE_TYPES])
if OVERWRITE_EVIDENCE:
    collect_evidence_cmd.append("--overwrite")

print(shell_join(collect_evidence_cmd))

In [ ]:
if RUN_COLLECT_EVIDENCE:
    run_command(collect_evidence_cmd, env=RUN_ENV)
else:
    print("Skipping collect_met_dx_notes.py")

In [ ]:
run_extraction_cmd = [
    PYTHON,
    "tasks/met_diagnosis/build_met_dx_labels.py",
    "--provider",
    PROVIDER,
    "--output-dir",
    OUTPUT_DIR,
    "--evidence-path",
    EVIDENCE_PATH,
    "--model",
    MODEL_NAME,
    "--max-workers",
    str(MAX_WORKERS),
    "--max-retries",
    str(MAX_RETRIES),
]

if MRNS:
    run_extraction_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
if MRN_FILE is not None:
    run_extraction_cmd.extend(["--mrn-file", MRN_FILE])
if LIMIT_PATIENTS is not None:
    run_extraction_cmd.extend(["--limit-patients", str(LIMIT_PATIENTS)])
if REBUILD_LABELS_ONLY and OVERWRITE:
    raise ValueError("REBUILD_LABELS_ONLY cannot be combined with OVERWRITE")
if REBUILD_LABELS_ONLY:
    run_extraction_cmd.append("--rebuild-labels-only")
if OVERWRITE:
    run_extraction_cmd.append("--overwrite")

print(shell_join(run_extraction_cmd))

In [ ]:
if RUN_EXTRACTION:
    run_command(run_extraction_cmd, env=RUN_ENV)
else:
    print("Skipping build_met_dx_labels.py")

In [ ]:
# Rerun only patients not yet marked "ok" in the processed-patients log.
# (These pipelines do not track a separate failure artifact — rerunning without
# --overwrite skips already-completed patients automatically.)
retry_cmd = [part for part in run_extraction_cmd if part != "--overwrite"]
print(shell_join(retry_cmd))

if RUN_RETRY_FAILURES:
    run_command(retry_cmd, env=RUN_ENV)
else:
    print("Skipping retry")

## Review

For a precision-biased task the review below is the point, not an afterthought. Read every positive's
`supporting_quote` by hand — each must be a self-evidently metastatic sentence. Then histogram the
rejection reasons: that column is the tuning signal for `tasks/met_diagnosis/veto.py`.

What a healthy histogram looks like:

- A large `negated_or_hedged:no` share is **expected**. A surveillance imaging report repeats "no
  evidence of metastatic disease" every few months for years, and the collector retrieves all of it.
- `regional_nodal_only` firing on genuine bone-met quotes, or `no_metastatic_assertion` firing on
  clean imaging impressions, means the gate needs a pattern fix.
- Positives whose `met_site` looks wrong for the quote are worth checking — the site is corroborated
  against the grounded quote, so a mismatch there points at the site regexes rather than the model.

Changing the gate bumps `VETO_VERSION`, which is part of the run fingerprint — so a gate change
requires re-running stage 2 with `OVERWRITE = True`.

Where `cancer_stage` output also exists, cross-checking is the highest-yield manual-review set:
patients with `stage_group == "IV"` or a non-empty `metastatic_sites` should be
`has_metastatic_disease == True`, and disagreements in either direction are worth reading.


In [ ]:
import polars as pl

labels = pl.read_parquet(Path(OUTPUT_DIR) / "met_dx_labels.parquet")
print(f"{labels.height} patients labeled")
positives = labels.filter(pl.col("has_metastatic_disease")).height
print(f"{positives} positive ({positives / max(labels.height, 1):.1%})")
display(
    labels.group_by(["has_metastatic_disease", "date_source", "label_source"])
    .len()
    .sort("len", descending=True)
)
display(labels.group_by("met_site").len().sort("len", descending=True))

In [ ]:
# Every positive, for manual review of the supporting quote.
display(
    labels.filter(pl.col("has_metastatic_disease"))
    .select([
        "DFCI_MRN", "first_metastasis_date", "date_source", "evidence_type",
        "assertion_type", "met_site", "modality", "confidence", "supporting_quote",
    ])
    .sort("first_metastasis_date")
    .head(25)
)

In [ ]:
# Rejection histogram -- the tuning signal for veto.py.
rejected_path = Path(OUTPUT_DIR) / "met_dx_rejected_findings.parquet"
if rejected_path.exists():
    rejected = pl.read_parquet(rejected_path)
    display(rejected.group_by(["stage", "reason"]).len().sort("len", descending=True))
    # Spot-check the two reasons most likely to be over-rejecting.
    for prefix in ("regional_nodal_only", "no_metastatic_assertion"):
        subset = rejected.filter(pl.col("reason") == prefix)
        print(f"{prefix}: {subset.height} rejections")
        display(subset.select(["DFCI_MRN", "reason", "item_json"]).head(15))
else:
    print(f"No rejected-findings artifact yet: {rejected_path}")